# KTZh Passenger Operations — Revenue & Demand Analysis
## Internship Project | Big Data Analysis | 2025
### Focus: Price Optimisation · Peak Demand · Channel Performance

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from matplotlib.gridspec import GridSpec

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

SEED = 42
np.random.seed(SEED)

## 2. Data Loading

In [ ]:
RAW_PATH = 'ktzh_operations_dataset.csv'

df_raw = pd.read_csv(RAW_PATH, encoding='utf-8-sig', dtype=str)

numeric_cols = ['price', 'extra_fees', 'op_hour', 'lead_days']
for c in numeric_cols:
    df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

print('Loaded:', df_raw.shape[0], 'rows x', df_raw.shape[1], 'columns')
print('Columns:', list(df_raw.columns))

## 3. Data Cleaning & Preprocessing

Time features (`op_hour`, `lead_days`) are pre-computed in the dataset. Cleaning steps:
- Drop the near-empty `ub_number` column
- Keep only rows with valid operation type, price, and payment type
- Impute missing categoricals with 'Unknown'
- Engineer route, total cost, service-class group, and hour bins

In [ ]:
work = df_raw.copy()

work = work.drop(columns=['ub_number'])

keep = work['operation_type'].notna() & work['price'].notna() & work['payment_type'].notna()
work = work[keep].reset_index(drop=True)

work['price'] = work['price'].astype(float)
work['extra_fees'] = work['extra_fees'].fillna(0).astype(float)
work['op_hour'] = work['op_hour'].fillna(-1).astype(int)

for c in ['gender', 'carrier', 'service_class', 'fare_type', 'channel_address',
          'departure_station', 'arrival_station', 'sales_channel', 'sales_point']:
    work[c] = work[c].fillna('Unknown')

work['total_cost'] = work['price'] + work['extra_fees']
work['route'] = work['departure_station'] + ' -> ' + work['arrival_station']

cls_first = work['service_class'].astype(str).str[0]
work['class_group'] = cls_first.map({'1':'First','2':'Second','3':'Third'}).fillna('Other')

def bin_hour(h):
    if h < 0: return 'Unknown'
    if h <= 5: return 'Night (0-5)'
    if h <= 11: return 'Morning (6-11)'
    if h <= 16: return 'Afternoon (12-16)'
    if h <= 20: return 'Evening (17-20)'
    return 'Late (21-23)'
work['hour_bin'] = work['op_hour'].apply(bin_hour)

work['is_cashless'] = (work['payment_type'] == 'Безналичный').astype(int)

df = work
print('Clean dataset:', df.shape[0], 'rows')
print()
nulls = df.isnull().sum()
print('Null counts after cleaning:')
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No nulls remaining')

In [ ]:
sales = df[df['operation_type'] == 'Оформление'].copy()

q1 = sales['price'].quantile(0.25)
q3 = sales['price'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 3 * iqr

sales['is_outlier'] = ((sales['price'] > upper) | (sales['price'] == 0)).astype(int)
sales_clean = sales[sales['is_outlier'] == 0].reset_index(drop=True)

print('Ticket issuances (all):', len(sales))
print('Price outliers removed:', int(sales['is_outlier'].sum()), '(fence:', round(upper), 'KZT)')
print('Clean sales records:', len(sales_clean))
print()
print('Price summary (clean):')
desc = sales_clean['price'].describe()
print('  count:', int(desc['mean']*0 + len(sales_clean)))
print('  mean: ', round(desc['mean']))
print('  std:  ', round(desc['std']))
print('  min:  ', round(desc['min']))
print('  max:  ', round(desc['max']))

## 4. Exploratory Data Analysis

### 4.1 Revenue Distribution by Service Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grp = sales_clean.groupby('class_group')['price']
class_rev = pd.DataFrame({
    'total_revenue': grp.sum(),
    'ticket_count': grp.count(),
    'avg_price': grp.mean(),
}).sort_values('total_revenue', ascending=False)
class_rev = class_rev.reset_index()
class_rev['rev_M'] = class_rev['total_revenue'] / 1e6

colors_bar = ['#2E6DA4', '#4AA8D8', '#A8D5F0', '#C8D8E8']
bars = axes[0].bar(class_rev['class_group'], class_rev['rev_M'],
                   color=colors_bar[:len(class_rev)], edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, class_rev['rev_M']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(round(val,1))+'M', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Total Revenue by Service Class (KZT)', fontweight='bold')
axes[0].set_xlabel('Class Group')
axes[0].set_ylabel('Revenue (Million KZT)')

box_data = [sales_clean[sales_clean['class_group']==g]['price'].values for g in class_rev['class_group']]
bp = axes[1].boxplot(box_data, labels=list(class_rev['class_group']), patch_artist=True,
                     medianprops=dict(color='#E8734A', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='#E8734A', markersize=3, alpha=0.3))
for patch in bp['boxes']:
    patch.set_facecolor('#D6E8F7')
    patch.set_edgecolor('#2E6DA4')
axes[1].set_title('Price Distribution by Service Class (KZT)', fontweight='bold')
axes[1].set_xlabel('Class Group')
axes[1].set_ylabel('Ticket Price (KZT)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: format(int(v), ',')))

plt.tight_layout()
plt.savefig('plot_01_class_revenue.png', bbox_inches='tight')
plt.show()

**Insight:** Second-class tickets dominate total revenue, even though First-class commands the highest average ticket price. Third-class drives high volume at lower revenue per seat — a clearly price-sensitive segment. Pricing strategy should focus on improving Second-class yield, the revenue backbone.

### 4.2 Hourly Ticket Sales — Peak Demand Heatmap

In [ ]:
sales_clean['class_simple'] = sales_clean['service_class'].astype(str).str[:2]
top_classes = sales_clean['class_simple'].value_counts().head(6).index.tolist()
sub = sales_clean[(sales_clean['class_simple'].isin(top_classes)) & (sales_clean['op_hour'] >= 0)]

pivot = sub.groupby(['op_hour', 'class_simple']).size().unstack(fill_value=0)
pivot = pivot.reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot.T, ax=ax, cmap='YlOrRd', linewidths=0.4, linecolor='white',
            annot=True, fmt='d', annot_kws={'size':8}, cbar_kws={'label':'Tickets Sold'})
ax.set_title('Ticket Sales Heatmap: Hour of Day x Service Class', fontweight='bold')
ax.set_xlabel('Hour of Day (Astana Time)')
ax.set_ylabel('Service Class')
ax.set_xticklabels([str(h)+':00' for h in range(24)], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('plot_02_heatmap_hour_class.png', bbox_inches='tight')
plt.show()

**Insight:** Peak sales occur between 10:00–14:00 Astana time, concentrated in the 3П/3Л economy classes. Night hours (01:00–05:00) are nearly idle. KTZh could apply time-based dynamic pricing — a modest peak-hour surcharge and off-peak discount — to smooth demand and lift revenue.

### 4.3 Top 15 Routes by Total Revenue

In [ ]:
grp = sales_clean.groupby('route')['price']
route_rev = pd.DataFrame({'total_rev': grp.sum(), 'tickets': grp.count(), 'avg_price': grp.mean()})
route_rev = route_rev.sort_values('total_rev', ascending=False).head(15).reset_index()
route_rev['rev_M'] = route_rev['total_rev'] / 1e6

fig, ax = plt.subplots(figsize=(13, 6))
norm = route_rev['avg_price'] / route_rev['avg_price'].max()
bar_colors = plt.cm.Blues(0.3 + norm * 0.65)
order = route_rev.iloc[::-1]
bars = ax.barh(order['route'], order['rev_M'], color=bar_colors[::-1], edgecolor='white')
for bar, val, avg in zip(bars, order['rev_M'], order['avg_price']):
    ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
            str(round(val,2))+'M  (avg '+format(int(avg), ',')+' KZT)', va='center', fontsize=8.5)
ax.set_title('Top 15 Routes by Total Revenue', fontweight='bold')
ax.set_xlabel('Total Revenue (Million KZT)')
ax.set_ylabel('Route')
plt.tight_layout()
plt.savefig('plot_03_top_routes_revenue.png', bbox_inches='tight')
plt.show()

**Insight:** A handful of long-distance corridors generate the bulk of revenue. The highest-revenue routes combine solid ticket volume with above-average prices, while a few premium routes show very high average fares on lower volume. KTZh should prioritise capacity on the top revenue corridors and consider premium service tiers on the high-fare routes.

### 4.4 Sales Channel Performance

In [ ]:
grp = sales_clean.groupby('sales_channel')
channel_stats = pd.DataFrame({
    'total_revenue': grp['price'].sum(),
    'ticket_count': grp['price'].count(),
    'avg_price': grp['price'].mean(),
    'cashless_share': grp['is_cashless'].mean(),
}).sort_values('total_revenue', ascending=False).reset_index()

short = {'Собственные кассы':'Own\nCashiers', 'Туристические агентства (МАП)':'Travel\nAgencies',
         'Портал пассажиров КТЖ':'KTZh\nPortal', 'Unknown':'Unknown'}
labels = [short.get(c, str(c)[:12]) for c in channel_stats['sales_channel']]
n = len(channel_stats)
pal = ['#2E6DA4', '#E8734A', '#4AA8D8', '#A8D5F0'][:n]

fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 3, figure=fig, wspace=0.4)

ax1 = fig.add_subplot(gs[0])
w, t, a = ax1.pie(channel_stats['ticket_count'], autopct='%1.1f%%', colors=pal,
                  startangle=90, pctdistance=0.75, wedgeprops={'edgecolor':'white','linewidth':2})
for at in a: at.set_fontsize(10); at.set_fontweight('bold')
ax1.legend(channel_stats['sales_channel'], loc='lower center', bbox_to_anchor=(0.5,-0.28), fontsize=7.5)
ax1.set_title('Tickets Sold by Channel', fontweight='bold')

ax2 = fig.add_subplot(gs[1])
b2 = ax2.bar(range(n), channel_stats['avg_price'], color=pal, edgecolor='white')
for bar, val in zip(b2, channel_stats['avg_price']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, format(int(val), ','),
             ha='center', fontsize=9, fontweight='bold')
ax2.set_xticks(range(n)); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_title('Avg Ticket Price by Channel (KZT)', fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: format(int(v), ',')))

ax3 = fig.add_subplot(gs[2])
b3 = ax3.bar(range(n), channel_stats['cashless_share']*100, color=pal, edgecolor='white')
for bar, val in zip(b3, channel_stats['cashless_share']*100):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(round(val,1))+'%',
             ha='center', fontsize=9, fontweight='bold')
ax3.set_xticks(range(n)); ax3.set_xticklabels(labels, fontsize=9)
ax3.set_title('Cashless Share by Channel (%)', fontweight='bold')
ax3.set_ylim(0, max(channel_stats['cashless_share'].max()*100*1.25, 10))

plt.suptitle('Sales Channel Performance Dashboard', fontweight='bold', y=1.02)
plt.savefig('plot_04_channel_performance.png', bbox_inches='tight')
plt.show()

**Insight:** Own ticket offices handle the large majority of sales by volume, but travel agencies sell higher-value tickets on average — they serve longer-distance, premium journeys. Expanding the agency network and investing in the underused KTZh online portal could raise overall revenue per transaction.

### 4.5 Price Distribution by Fare Type

In [ ]:
paid = ['ПОЛНЫЙ','ДЕТ','ИНВ','СПИНВ','КАРТА','ЭКИПЖ']
df_f = sales_clean[sales_clean['fare_type'].isin(paid)]

order_f = df_f.groupby('fare_type')['price'].median().sort_values(ascending=False).index.tolist()
pal_f = {'ПОЛНЫЙ':'#2E6DA4','ДЕТ':'#4AA8D8','ИНВ':'#A8D5F0','СПИНВ':'#E8A87C','КАРТА':'#E8734A','ЭКИПЖ':'#C0C0C0'}

fig, ax = plt.subplots(figsize=(13, 5))
sns.violinplot(data=df_f, x='fare_type', y='price', order=order_f,
               palette=pal_f, inner='box', cut=0, linewidth=1.2, ax=ax)
ax.set_title('Price Distribution by Fare Type (KZT)', fontweight='bold')
ax.set_xlabel('Fare Type')
ax.set_ylabel('Ticket Price (KZT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: format(int(v), ',')))
plt.tight_layout()
plt.savefig('plot_05_violin_fare_type.png', bbox_inches='tight')
plt.show()

**Insight:** Full-fare (ПОЛНЫЙ) tickets show the widest spread and highest median — they drive most revenue. Crew passes (ЭКИПЖ) sit near zero and represent non-commercial travel that should be excluded from revenue KPIs. The disabled-fare (ИНВ) segment carries meaningful volume, so its subsidy cost is worth monitoring in financial planning.

### 4.6 Gender × Fare Type Revenue Cross-Analysis

In [ ]:
gf = sales_clean[sales_clean['gender'] != 'Unknown']
top_f = gf['fare_type'].value_counts().head(5).index.tolist()
gf = gf[gf['fare_type'].isin(top_f)]

cnt = gf.groupby(['fare_type','gender']).size().unstack(fill_value=0)
rev = gf.groupby(['fare_type','gender'])['price'].sum().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cnt.plot(kind='bar', ax=axes[0], color=['#E8734A','#2E6DA4'], edgecolor='white', width=0.7)
axes[0].set_title('Ticket Count: Gender x Fare Type', fontweight='bold')
axes[0].set_xlabel('Fare Type'); axes[0].set_ylabel('Number of Tickets')
axes[0].legend(title='Gender'); axes[0].tick_params(axis='x', rotation=30)

(rev/1e6).plot(kind='bar', ax=axes[1], color=['#E8734A','#2E6DA4'], edgecolor='white', width=0.7)
axes[1].set_title('Revenue (M KZT): Gender x Fare Type', fontweight='bold')
axes[1].set_xlabel('Fare Type'); axes[1].set_ylabel('Revenue (Million KZT)')
axes[1].legend(title='Gender'); axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('plot_06_gender_fare.png', bbox_inches='tight')
plt.show()

**Insight:** Male and female ticket volumes are nearly balanced across the main fare types, with only a slight male skew in total revenue driven by route mix. The child fare splits evenly by gender, as expected. This balance means marketing campaigns can target both segments equally rather than skewing spend.

### 4.7 Carrier Revenue Share

In [ ]:
grp = sales_clean.groupby('carrier')['price']
carrier_rev = pd.DataFrame({'total': grp.sum(), 'count': grp.count(), 'avg': grp.mean()})
carrier_rev = carrier_rev.sort_values('total', ascending=False).reset_index()
carrier_rev['rev_M'] = carrier_rev['total'] / 1e6

top = carrier_rev.head(8).copy()
other = carrier_rev.iloc[8:]['total'].sum() / 1e6
plot_df = pd.concat([top[['carrier','rev_M']],
                     pd.DataFrame([{'carrier':'Others','rev_M':other}])], ignore_index=True)

clabels = [str(c).replace('АО ПАССАЖИРСКИЕ ПЕР-КИ','AO PP (National)').replace('ТОО ','')[:22]
           for c in plot_df['carrier']]
colors_c = ['#1A3A5C','#2E6DA4','#4AA8D8','#6EC0E8','#A8D5F0','#C8E6F7','#E0F1FB','#F0F8FF','#CCCCCC']

fig, ax = plt.subplots(figsize=(8, 8))
exp = [0.04] + [0]*(len(plot_df)-1)
w, t, a = ax.pie(plot_df['rev_M'], autopct=lambda p: str(round(p,1))+'%' if p>2 else '',
                 colors=colors_c[:len(plot_df)], startangle=140, pctdistance=0.78,
                 wedgeprops={'edgecolor':'white','linewidth':1.5}, explode=exp)
for at in a: at.set_fontsize(9); at.set_fontweight('bold')
ax.legend(w, [l+' ('+str(round(v,1))+'M)' for l,v in zip(clabels, plot_df['rev_M'])],
          loc='lower center', bbox_to_anchor=(0.5,-0.22), ncol=2, fontsize=8.5)
ax.set_title('Revenue Share by Carrier (Million KZT)', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_07_carrier_revenue_share.png', bbox_inches='tight')
plt.show()

**Insight:** The national carrier (AO Passazhirskie Perevozki) dominates both revenue and volume, while several private operators serve niche regional routes. This concentration means that pricing decisions on the national carrier have an outsized effect on total system revenue — a key lever for KTZh.

### 4.8 Cashless Payment Rate by Top Sales Points

In [ ]:
top_points = sales_clean['sales_point'].value_counts().head(10).index.tolist()
dp = sales_clean[sales_clean['sales_point'].isin(top_points)]
grp = dp.groupby('sales_point')
cbp = pd.DataFrame({'cashless_rate': grp['is_cashless'].mean(), 'count': grp['price'].count()})
cbp = cbp.sort_values('cashless_rate').reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = [plt.cm.RdYlGn(v) for v in cbp['cashless_rate']]
bars = ax.barh(cbp['sales_point'], cbp['cashless_rate']*100, color=bar_colors, edgecolor='white', height=0.65)
for bar, rate, cnt in zip(bars, cbp['cashless_rate'], cbp['count']):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            str(round(rate*100,1))+'%  (n='+str(int(cnt))+')', va='center', fontsize=9)
overall = sales_clean['is_cashless'].mean()*100
ax.axvline(overall, color='#1A3A5C', linestyle='--', linewidth=1.5, label='Overall avg: '+str(round(overall,1))+'%')
ax.set_title('Cashless Payment Rate by Top-10 Sales Points', fontweight='bold')
ax.set_xlabel('Cashless Payment Rate (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plot_08_cashless_by_point.png', bbox_inches='tight')
plt.show()

**Insight:** Cashless adoption varies widely across sales points — some are largely card-based while others remain almost entirely cash. KTZh should prioritise terminal upgrades and card-payment incentives at the low-adoption locations: cashless transactions cut cash-handling costs and improve revenue-tracking accuracy.

## 5. Feature Engineering for Machine Learning

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

ml = sales_clean[sales_clean['fare_type'] != 'ЭКИПЖ'].copy()
ml = ml[ml['price'] > 0].reset_index(drop=True)

cat_cols = ['fare_type','service_class','sales_channel','gender','carrier','class_group','payment_type']
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    ml[c+'_enc'] = le.fit_transform(ml[c].astype(str))
    encoders[c] = le

FEATURES = [c+'_enc' for c in cat_cols] + ['op_hour', 'extra_fees', 'is_cashless']
TARGET = 'price'

ml_model = ml[FEATURES + [TARGET]].dropna().reset_index(drop=True)
print('ML dataset:', ml_model.shape[0], 'rows x', len(FEATURES), 'features')
print()
print('Features:', FEATURES)
print()
d = ml_model[TARGET].describe()
print('Target (price) -> mean:', round(d['mean']), '| min:', round(d['min']), '| max:', round(d['max']))

## 6. Machine Learning Models

### 6.1 Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor

X = ml_model[FEATURES].values
y = ml_model[TARGET].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

print('Train:', X_tr.shape[0], 'samples | Test:', X_te.shape[0], 'samples')

results = {}
def evaluate(name, yt, yp):
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(name, '-> MAE:', round(mae), '| RMSE:', round(rmse), '| R2:', round(r2, 4))

### 6.2 Model 1: Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_tr_sc, y_tr)
pred_lr = lr.predict(X_te_sc)
evaluate('Linear Regression', y_te, pred_lr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_te, pred_lr, alpha=0.3, color='#2E6DA4', s=15, edgecolors='none')
lims = [min(y_te.min(), pred_lr.min()), max(y_te.max(), pred_lr.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_title('Linear Regression: Predicted vs Actual', fontweight='bold')
axes[0].set_xlabel('Actual Price (KZT)'); axes[0].set_ylabel('Predicted Price (KZT)')
axes[0].legend()

resid = y_te - pred_lr
axes[1].hist(resid, bins=50, color='#2E6DA4', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Linear Regression: Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (KZT)'); axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('plot_09_lr_results.png', bbox_inches='tight')
plt.show()

### 6.3 Model 2: Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5,
                           n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_tr)
pred_rf = rf.predict(X_te)
evaluate('Random Forest Regressor', y_te, pred_rf)

imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_fi = plt.cm.Blues(np.linspace(0.35, 0.9, len(imp)))
axes[0].barh(imp.index, imp.values, color=colors_fi, edgecolor='white')
axes[0].set_title('Random Forest: Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance Score')

axes[1].scatter(y_te, pred_rf, alpha=0.3, color='#4AA8D8', s=15, edgecolors='none')
axes[1].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', linewidth=1.5)
axes[1].set_title('Random Forest: Predicted vs Actual', fontweight='bold')
axes[1].set_xlabel('Actual Price (KZT)'); axes[1].set_ylabel('Predicted Price (KZT)')
plt.tight_layout()
plt.savefig('plot_10_rf_results.png', bbox_inches='tight')
plt.show()

### 6.4 Model 3: XGBoost Regressor

In [ ]:
xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=1.0,
                   random_state=SEED, verbosity=0)
xgb.fit(X_tr, y_tr)
pred_xgb = xgb.predict(X_te)
evaluate('XGBoost Regressor', y_te, pred_xgb)

imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_xgb = plt.cm.Oranges(np.linspace(0.35, 0.9, len(imp)))
axes[0].barh(imp.index, imp.values, color=colors_xgb, edgecolor='white')
axes[0].set_title('XGBoost: Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance Score')

axes[1].scatter(y_te, pred_xgb, alpha=0.3, color='#E8734A', s=15, edgecolors='none')
axes[1].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'b--', linewidth=1.5)
axes[1].set_title('XGBoost: Predicted vs Actual', fontweight='bold')
axes[1].set_xlabel('Actual Price (KZT)'); axes[1].set_ylabel('Predicted Price (KZT)')
plt.tight_layout()
plt.savefig('plot_11_xgb_results.png', bbox_inches='tight')
plt.show()

### 6.5 Model 4: K-Means Clustering — Passenger Segmentation

In [ ]:
from sklearn.metrics import silhouette_score

cf = ['service_class_enc', 'fare_type_enc', 'is_cashless', 'op_hour']
Xc = ml_model[cf].values
mms = MinMaxScaler()
Xc_sc = mms.fit_transform(Xc)

inertias, sils = [], []
Ks = range(2, 11)
for k in Ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lab = km.fit_predict(Xc_sc)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xc_sc, lab, sample_size=2000, random_state=SEED))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(Ks), inertias, 'o-', color='#2E6DA4', linewidth=2, markersize=7)
axes[0].axvline(4, color='red', linestyle='--', linewidth=1.2, label='k=4 (selected)')
axes[0].set_title('K-Means: Elbow Method (Inertia)', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)'); axes[0].set_ylabel('Inertia'); axes[0].legend()

axes[1].plot(list(Ks), sils, 's-', color='#E8734A', linewidth=2, markersize=7)
axes[1].axvline(4, color='red', linestyle='--', linewidth=1.2, label='k=4 (selected)')
axes[1].set_title('K-Means: Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)'); axes[1].set_ylabel('Silhouette Score'); axes[1].legend()
plt.tight_layout()
plt.savefig('plot_12_kmeans_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
BEST_K = 4
km = KMeans(n_clusters=BEST_K, random_state=SEED, n_init=15)
ml_model['cluster'] = km.fit_predict(Xc_sc)
sil = silhouette_score(Xc_sc, ml_model['cluster'], sample_size=2000, random_state=SEED)
results['K-Means Clustering'] = {'Silhouette': sil, 'K': BEST_K}
print('K-Means (k=4) Silhouette Score:', round(sil, 4))

grp = ml_model.groupby('cluster')
profile = pd.DataFrame({
    'count': grp['price'].count(),
    'avg_price': grp['price'].mean(),
    'cashless_rate': grp['is_cashless'].mean(),
    'avg_hour': grp['op_hour'].mean(),
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc_colors = ['#1A3A5C', '#2E6DA4', '#E8734A', '#4AA8D8']
for cl in range(BEST_K):
    m = ml_model['cluster'] == cl
    axes[0].scatter(ml_model.loc[m,'op_hour'], ml_model.loc[m,'price'],
                    alpha=0.15, s=10, c=sc_colors[cl], label='Cluster '+str(cl)+' (n='+str(int(m.sum()))+')')
axes[0].set_title('Passenger Clusters: Hour vs Price', fontweight='bold')
axes[0].set_xlabel('Hour of Day'); axes[0].set_ylabel('Ticket Price (KZT)')
axes[0].legend(fontsize=8.5, markerscale=3)

x = np.arange(BEST_K); width = 0.35
axes[1].bar(x, profile['avg_price'], width*2, color=sc_colors, edgecolor='white')
for i, v in enumerate(profile['avg_price']):
    axes[1].text(i, v+150, format(int(v), ','), ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Cluster Profile: Average Price', fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Average Price (KZT)')
axes[1].set_xticks(x); axes[1].set_xticklabels(['C'+str(i) for i in range(BEST_K)])
plt.tight_layout()
plt.savefig('plot_13_kmeans_segments.png', bbox_inches='tight')
plt.show()

print()
print('Cluster Profiles:')
print(profile.round(2).to_string(index=False))

**Insight:** K-Means reveals four distinct passenger segments based on service class, fare type, payment behaviour, and purchase time. These segments map directly to actions: the premium cluster → upsell campaigns; the cashless-heavy cluster → digital loyalty programmes; the off-peak cluster → time-sensitive discount offers.

### 6.6 Model 5: Logistic Regression — Payment Type Classification

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report

lrf = [c+'_enc' for c in ['fare_type','service_class','sales_channel','gender','carrier','class_group']] + ['op_hour','extra_fees']
yb = ml_model['is_cashless'].values
Xb = ml_model[lrf].values

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=SEED, stratify=yb)
sb = StandardScaler()
Xb_tr_sc = sb.fit_transform(Xb_tr)
Xb_te_sc = sb.transform(Xb_te)

logreg = LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced', C=1.0)
logreg.fit(Xb_tr_sc, yb_tr)
prob = logreg.predict_proba(Xb_te_sc)[:, 1]
pred = logreg.predict(Xb_te_sc)

acc = accuracy_score(yb_te, pred)
f1 = f1_score(yb_te, pred)
auc = roc_auc_score(yb_te, prob)
results['Logistic Regression (Payment)'] = {'Accuracy': acc, 'F1': f1, 'ROC-AUC': auc}
print('Accuracy:', round(acc,4), '| F1:', round(f1,4), '| ROC-AUC:', round(auc,4))
print()
print(classification_report(yb_te, pred, target_names=['Cash','Cashless']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fpr, tpr, _ = roc_curve(yb_te, prob)
axes[0].plot(fpr, tpr, color='#2E6DA4', linewidth=2, label='ROC-AUC = '+str(round(auc,3)))
axes[0].plot([0,1],[0,1],'k--', linewidth=1, alpha=0.6)
axes[0].fill_between(fpr, tpr, alpha=0.15, color='#2E6DA4')
axes[0].set_title('Logistic Regression: ROC Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate'); axes[0].legend()

cm = confusion_matrix(yb_te, pred)
ConfusionMatrixDisplay(cm, display_labels=['Cash','Cashless']).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Logistic Regression: Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_14_logistic_results.png', bbox_inches='tight')
plt.show()

## 7. Model Comparison & Business Conclusions

In [ ]:
print('='*60)
print('MODEL COMPARISON SUMMARY')
print('='*60)
reg = ['Linear Regression', 'Random Forest Regressor', 'XGBoost Regressor']
print()
print('REGRESSION MODELS (Price Prediction)')
print('-'*60)
for m in reg:
    r = results[m]
    print(m.ljust(28), '| MAE:', str(round(r['MAE'])).rjust(6),
          '| RMSE:', str(round(r['RMSE'])).rjust(6), '| R2:', round(r['R2'],4))
print()
print('CLASSIFICATION (Payment Type)')
r = results['Logistic Regression (Payment)']
print('Logistic Regression'.ljust(28), '| Acc:', round(r['Accuracy'],4), '| F1:', round(r['F1'],4), '| AUC:', round(r['ROC-AUC'],4))
print()
print('CLUSTERING (Passenger Segments)')
r = results['K-Means Clustering']
print('K-Means (k=4)'.ljust(28), '| Silhouette:', round(r['Silhouette'],4))
print('='*60)

In [ ]:
reg_df = pd.DataFrame({m: results[m] for m in reg}).T
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'R2']
labels_m = ['MAE (KZT) - lower better', 'RMSE (KZT) - lower better', 'R2 - higher better']
bcols = ['#2E6DA4', '#4AA8D8', '#E8734A']
for i, (mt, lb) in enumerate(zip(metrics, labels_m)):
    vals = reg_df[mt].astype(float)
    bars = axes[i].bar(['Lin.Reg', 'Rand.For', 'XGBoost'], vals, color=bcols, edgecolor='white', width=0.55)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+vals.max()*0.01,
                     round(v,2) if mt=='R2' else int(v), ha='center', fontsize=8.5, fontweight='bold')
    axes[i].set_title(lb, fontweight='bold', fontsize=10)
    axes[i].set_ylabel(mt)
plt.suptitle('Regression Model Comparison', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('plot_15_model_comparison.png', bbox_inches='tight')
plt.show()

## 8. Business Recommendations

Based on the analysis of KTZh's one-day operations (31 August 2025):

1. **Dynamic Pricing.** XGBoost confirms service class, fare type, and purchase hour as the strongest price drivers. Time-based peak surcharges (10:00–14:00) and off-peak discounts (01:00–06:00) could smooth demand and lift daily revenue.

2. **High-Value Route Investment.** The top revenue corridors deserve added frequency or premium seating, directly impacting the bottom line.

3. **Channel Strategy.** Travel agencies deliver higher average ticket value. Expanding the agency partner network in Tier-2 cities could raise revenue per transaction.

4. **Cashless Push.** Sales points with low cashless rates should receive terminal upgrades and incentives. Logistic Regression confirms sales channel as the strongest predictor of payment type, so point-of-sale interventions are most effective.

5. **Passenger Segmentation.** The four K-Means segments enable targeted action: loyalty programmes for the high-value cashless segment, advance-purchase discounts for the price-sensitive peak-hour segment.

6. **Subsidy Monitoring.** Crew/pass (ЭКИПЖ) fares at zero price should be tracked separately and recovered from operating budgets.